# ⚙️ Predictive Gearbox Vibration Control Using a 1D Convolutional Neural Network

Forecast vibration 30 seconds ahead from a 1000-sample vibration waveform plus RPM, torque, temperature, and motor current.

> The 7 mm/s danger threshold and all responses are educational simulation assumptions.

👉 **Open the interactive companion:** [https://gearbox-vibration-prediction.streamlit.app](https://gearbox-vibration-prediction.streamlit.app/?stage=start)

## Complete workflow

Waveform → 1D CNN features + operating values → fusion network → future vibration → project condition band → simulated response.

## Interactive learning journey

- [Vibration Before Failure](https://gearbox-vibration-prediction.streamlit.app/?stage=problem) — Predictive Condition Monitoring
- [Five Sources of Evidence](https://gearbox-vibration-prediction.streamlit.app/?stage=inputs) — Multimodal Inputs
- [Inside the Vibration Trace](https://gearbox-vibration-prediction.streamlit.app/?stage=waveform) — Time-Domain Signal
- [Simulated Degradation Runs](https://gearbox-vibration-prediction.streamlit.app/?stage=data) — Training Dataset
- [Preparing Signals and Operating Data](https://gearbox-vibration-prediction.streamlit.app/?stage=prepare) — Normalization
- [Combining Vibration and Load](https://gearbox-vibration-prediction.streamlit.app/?stage=fusion) — CNN Feature Fusion
- [Vibration Thirty Seconds Ahead](https://gearbox-vibration-prediction.streamlit.app/?stage=forecast) — Regression Forecast
- [Project Condition Bands](https://gearbox-vibration-prediction.streamlit.app/?stage=decision) — Decision Layer
- [Wait Versus Respond Early](https://gearbox-vibration-prediction.streamlit.app/?stage=compare) — Intervention Simulation and Audit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error,confusion_matrix,ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import layers,Model
from tensorflow.keras.callbacks import EarlyStopping
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
N=1000;THRESHOLD=7.0

---
# 1. Vibration Before Failure
### Phase 1 of 6 · The Developing Fault

## Part 1 · At the gearbox
Gear wear, damaged teeth, misalignment, imbalance, and resonance can make gearbox vibration grow during operation.

## Part 2 · The engineering challenge
A conventional alarm reacts after measured vibration already crosses its setpoint, leaving little time to change operating conditions.

## Part 3 · Where the AI comes in
Forecast vibration thirty seconds ahead so a simulated response can be evaluated before the threshold is crossed.

**Mechanical Engineering:** Vibration Before Failure → **AI:** Predictive Condition Monitoring → `forecast rather than wait for threshold`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=problem](https://gearbox-vibration-prediction.streamlit.app/?stage=problem)

## Part 4 · The technical explanation

The target is continuous future vibration in mm/s. Normal, Warning, and Dangerous are calculated afterward so the model and project operating assumption remain separate.

## Part 5 · What you just built

**In the notebook:** Define current and future vibration, forecast horizon, and the simulated danger threshold.

**Takeaway:** Prediction creates warning time; it does not diagnose every mechanical root cause.

[Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Five Sources of Evidence](https://gearbox-vibration-prediction.streamlit.app/?stage=inputs) ▶

---
# 2. Five Sources of Evidence
### Phase 2 of 6 · Measuring the Gearbox

## Part 1 · At the gearbox
The waveform contains rotating and gear-mesh signatures while RPM, torque, temperature, and motor current describe operating state.

## Part 2 · The engineering challenge
The same vibration pattern can mean different risk near resonance or under a different load.

## Part 3 · Where the AI comes in
Let the CNN read the waveform and fuse its learned features with four normalized operating values.

**Mechanical Engineering:** Five Sources of Evidence → **AI:** Multimodal Inputs → `1000 vibration samples + RPM, torque, temperature, current`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=inputs](https://gearbox-vibration-prediction.streamlit.app/?stage=inputs)

## Part 4 · The technical explanation

In [ ]:
example={"rpm":1800,"torque_nm":42,"current_vibration":4.8,"temperature_c":68,"motor_current_a":11.2}
pd.Series(example,name="Current operating state")

## Part 5 · What you just built

**In the notebook:** Create waveform and operating-data arrays as two model inputs.

**Takeaway:** Signal shape and operating context belong together.

◀ [Previous: Vibration Before Failure](https://gearbox-vibration-prediction.streamlit.app/?stage=problem) &nbsp;|&nbsp; [Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Inside the Vibration Trace](https://gearbox-vibration-prediction.streamlit.app/?stage=waveform) ▶

---
# 3. Inside the Vibration Trace
### Phase 2 of 6 · Measuring the Gearbox

## Part 1 · At the gearbox
Gear rotation produces shaft harmonics, gear-mesh frequency, sidebands, impacts, and noise.

## Part 2 · The engineering challenge
One RMS value hides when impacts occur and how frequency-related patterns evolve within the sample window.

## Part 3 · Where the AI comes in
Preserve the short waveform so Conv1D filters can learn local oscillations and impulsive changes directly.

**Mechanical Engineering:** Inside the Vibration Trace → **AI:** Time-Domain Signal → `1000 acceleration/amplitude samples`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=waveform](https://gearbox-vibration-prediction.streamlit.app/?stage=waveform)

## Part 4 · The technical explanation

In [ ]:
def waveform(rpm,torque,wear,resonance,seed):
 rng=np.random.default_rng(seed);t=np.linspace(0,.5,N);shaft=rpm/60;mesh=shaft*12;amp=.7+1.5*wear+1.8*resonance+torque/90
 x=amp*np.sin(2*np.pi*shaft*t)+.32*amp*np.sin(2*np.pi*mesh*t)+.18*amp*np.sin(2*np.pi*(mesh-shaft)*t)
 impacts=(rng.random(N)<wear*.018)*rng.normal(0,3*amp,N);x+=np.convolve(impacts,np.exp(-np.arange(20)/5),mode="same")+.12*rng.normal(size=N);return t,x.astype("float32")
fig,ax=plt.subplots(1,3,figsize=(15,3))
for a,(w,r,title) in zip(ax,[(.1,.1,"Normal"),(.5,.5,"Developing"),(.9,.9,"Dangerous")]):
 t,x=waveform(1800,42,w,r,int(w*100));a.plot(t,x,linewidth=.6);a.set_title(title);a.grid(alpha=.2)
plt.show()

## Part 5 · What you just built

**In the notebook:** Plot healthy, developing, and dangerous simulated waveforms.

**Takeaway:** The waveform retains evidence discarded by a single vibration number.

◀ [Previous: Five Sources of Evidence](https://gearbox-vibration-prediction.streamlit.app/?stage=inputs) &nbsp;|&nbsp; [Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Simulated Degradation Runs](https://gearbox-vibration-prediction.streamlit.app/?stage=data) ▶

---
# 4. Simulated Degradation Runs
### Phase 2 of 6 · Measuring the Gearbox

## Part 1 · At the gearbox
Controlled condition-monitoring tests vary speed, load, temperature, wear, and proximity to resonance.

## Part 2 · The engineering challenge
A classroom notebook needs labelled future vibration without operating a real gearbox to failure.

## Part 3 · Where the AI comes in
Generate physically motivated synthetic runs and calculate a future amplitude from degradation and operating conditions.

**Mechanical Engineering:** Simulated Degradation Runs → **AI:** Training Dataset → `wear, resonance, tooth impacts, load variation`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=data](https://gearbox-vibration-prediction.streamlit.app/?stage=data)

## Part 4 · The technical explanation

In [ ]:
rng=np.random.default_rng(SEED);waves=[];ops=[];targets=[]
for k in range(2400):
 rpm=rng.uniform(700,2900);torque=rng.uniform(8,85);temp=rng.uniform(35,95);motor=2.5+.18*torque+rng.normal(0,.7);wear=rng.uniform(0,1);res=np.exp(-((rpm-rng.choice([1500,2100]))/260)**2);current=max(.5,.7+.0011*rpm+.022*torque+.025*(temp-45)+1.6*wear+1.4*res+rng.normal(0,.35));t,x=waveform(rpm,torque,wear,res,k);future=max(.5,current+.00008*torque*rpm+.020*(temp-50)+1.3*wear+1.8*res-2.7+rng.normal(0,.3));waves.append(x);ops.append([rpm,torque,temp,motor]);targets.append(future)
Xw=np.array(waves)[...,None];Xo=np.array(ops,dtype="float32");y=np.array(targets,dtype="float32")
print(Xw.shape,Xo.shape,y.shape);print("Future range:",y.min(),y.max())

## Part 5 · What you just built

**In the notebook:** Create balanced normal, warning, and dangerous examples with continuous future targets.

**Takeaway:** Synthetic data demonstrates the pipeline but cannot validate real machinery.

◀ [Previous: Inside the Vibration Trace](https://gearbox-vibration-prediction.streamlit.app/?stage=waveform) &nbsp;|&nbsp; [Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Preparing Signals and Operating Data](https://gearbox-vibration-prediction.streamlit.app/?stage=prepare) ▶

---
# 5. Preparing Signals and Operating Data
### Phase 3 of 6 · Learning Vibration Patterns

## Part 1 · At the gearbox
RPM is in thousands, torque in newton-metres, temperature in degrees, current in amperes, and vibration has its own scale.

## Part 2 · The engineering challenge
Raw units distort optimization, while fitting scalers on test records leaks future information.

## Part 3 · Where the AI comes in
Normalize waveform and operating features using training-set parameters only while keeping the physical target in mm/s.

**Mechanical Engineering:** Preparing Signals and Operating Data → **AI:** Normalization → `training-only channel and metadata scalers`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=prepare](https://gearbox-vibration-prediction.streamlit.app/?stage=prepare)

## Part 4 · The technical explanation

In [ ]:
indices=np.arange(len(y));tr,tmp=train_test_split(indices,test_size=.30,random_state=SEED);va,te=train_test_split(tmp,test_size=.50,random_state=SEED)
wave_scaler=StandardScaler().fit(Xw[tr].reshape(-1,1));op_scaler=StandardScaler().fit(Xo[tr])
def sw(a):return wave_scaler.transform(a.reshape(-1,1)).reshape(a.shape)
Xw_train,Xw_val,Xw_test=sw(Xw[tr]),sw(Xw[va]),sw(Xw[te]);Xo_train,Xo_val,Xo_test=op_scaler.transform(Xo[tr]),op_scaler.transform(Xo[va]),op_scaler.transform(Xo[te]);y_train,y_val,y_test=y[tr],y[va],y[te]
print(Xw_train.shape,Xo_train.shape)

## Part 5 · What you just built

**In the notebook:** Split first, then fit separate waveform and operating-feature scalers.

**Takeaway:** Preprocessing must preserve physical output units and avoid test leakage.

◀ [Previous: Simulated Degradation Runs](https://gearbox-vibration-prediction.streamlit.app/?stage=data) &nbsp;|&nbsp; [Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Combining Vibration and Load](https://gearbox-vibration-prediction.streamlit.app/?stage=fusion) ▶

---
# 6. Combining Vibration and Load
### Phase 3 of 6 · Learning Vibration Patterns

## Part 1 · At the gearbox
An impact pattern under high torque near resonance deserves different concern from the same pattern at light load far from resonance.

## Part 2 · The engineering challenge
A waveform-only model lacks context; a tabular-only model misses tooth impacts and harmonic structure.

## Part 3 · Where the AI comes in
Extract waveform features with Conv1D, encode operating values with Dense layers, and concatenate both representations.

**Mechanical Engineering:** Combining Vibration and Load → **AI:** CNN Feature Fusion → `Conv1D branch + metadata branch -> concatenate`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=fusion](https://gearbox-vibration-prediction.streamlit.app/?stage=fusion)

## Part 4 · The technical explanation

In [ ]:
win=layers.Input((1000,1),name="vibration_waveform");x=layers.Conv1D(32,11,activation="relu")(win);x=layers.MaxPooling1D(2)(x);x=layers.Conv1D(64,7,activation="relu")(x);x=layers.GlobalAveragePooling1D()(x)
oin=layers.Input((4,),name="operating_values");o=layers.Dense(16,activation="relu")(oin);f=layers.Concatenate()([x,o]);f=layers.Dense(48,activation="relu")(f);f=layers.Dropout(.15)(f);out=layers.Dense(1)(f);model=Model([win,oin],out);model.compile(optimizer="adam",loss="mae",metrics=["mae"]);model.summary()

## Part 5 · What you just built

**In the notebook:** Build the two-input fusion network.

**Takeaway:** Fusion lets learned signal patterns be interpreted in their operating context.

◀ [Previous: Preparing Signals and Operating Data](https://gearbox-vibration-prediction.streamlit.app/?stage=prepare) &nbsp;|&nbsp; [Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Vibration Thirty Seconds Ahead](https://gearbox-vibration-prediction.streamlit.app/?stage=forecast) ▶

---
# 7. Vibration Thirty Seconds Ahead
### Phase 4 of 6 · Forecasting Ahead

## Part 1 · At the gearbox
Maintenance decisions benefit from a physical forecast that can be compared with equipment-specific limits.

## Part 2 · The engineering challenge
Predicting only a class hides how close the gearbox is to a boundary and prevents a different project limit from being applied.

## Part 3 · Where the AI comes in
Train the network to output future vibration amplitude; classify it afterward with an explicit rule.

**Mechanical Engineering:** Vibration Thirty Seconds Ahead → **AI:** Regression Forecast → `one continuous output in mm/s`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=forecast](https://gearbox-vibration-prediction.streamlit.app/?stage=forecast)

## Part 4 · The technical explanation

In [ ]:
early=EarlyStopping(monitor="val_loss",patience=6,restore_best_weights=True);history=model.fit([Xw_train,Xo_train],y_train,validation_data=([Xw_val,Xo_val],y_val),epochs=45,batch_size=48,callbacks=[early],verbose=0)
pred=model.predict([Xw_test,Xo_test],verbose=0).ravel();print("MAE:",mean_absolute_error(y_test,pred));print("RMSE:",mean_squared_error(y_test,pred)**.5)
plt.figure(figsize=(6,6));plt.scatter(y_test,pred,s=10,alpha=.35);plt.plot([0,12],[0,12],"r--");plt.xlabel("Actual future mm/s");plt.ylabel("Predicted future mm/s");plt.grid(alpha=.2);plt.show()

## Part 5 · What you just built

**In the notebook:** Train with MAE loss and plot actual versus predicted future vibration.

**Takeaway:** Predict the physical quantity first and apply the operating rule separately.

◀ [Previous: Combining Vibration and Load](https://gearbox-vibration-prediction.streamlit.app/?stage=fusion) &nbsp;|&nbsp; [Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Project Condition Bands](https://gearbox-vibration-prediction.streamlit.app/?stage=decision) ▶

---
# 8. Project Condition Bands
### Phase 5 of 6 · Simulated Response

## Part 1 · At the gearbox
The demonstration needs transparent bands for when to continue, reduce speed, or inspect.

## Part 2 · The engineering challenge
A universal 7 mm/s claim would be inappropriate because allowable vibration depends on machine, measurement, standard, and specification.

## Part 3 · Where the AI comes in
Declare 7 mm/s as this simulation's danger assumption, with a warning band below it, and keep recommendations simulated.

**Mechanical Engineering:** Project Condition Bands → **AI:** Decision Layer → `Normal / Warning / Dangerous`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=decision](https://gearbox-vibration-prediction.streamlit.app/?stage=decision)

## Part 4 · The technical explanation

In [ ]:
def condition(v):return "NORMAL" if v<5 else "WARNING" if v<THRESHOLD else "DANGEROUS"
def response(c):return {"NORMAL":"Continue simulated operation","WARNING":"Simulated response: reduce RPM slightly","DANGEROUS":"Simulated response: move away from speed / stop for inspection"}[c]
i=8;print(f"Current vibration: {y_test[i]-1.5:.1f} mm/s");print(f"Predicted after 30 s: {pred[i]:.1f} mm/s");print("Condition:",condition(pred[i]));print(response(condition(pred[i])))

## Part 5 · What you just built

**In the notebook:** Convert predicted vibration into a condition and simulated response.

**Takeaway:** The threshold is a project assumption, not a universal machinery limit.

◀ [Previous: Vibration Thirty Seconds Ahead](https://gearbox-vibration-prediction.streamlit.app/?stage=forecast) &nbsp;|&nbsp; [Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Wait Versus Respond Early](https://gearbox-vibration-prediction.streamlit.app/?stage=compare) ▶

---
# 9. Wait Versus Respond Early
### Phase 6 of 6 · Engineering Audit

## Part 1 · At the gearbox
If resonance drives the rise, moving away from the operating speed may reduce excitation before the alarm point.

## Part 2 · The engineering challenge
An intervention can also reduce production and may not address tooth damage, looseness, or another root cause.

## Part 3 · Where the AI comes in
Simulate both trajectories, calculate peak vibration, time above threshold, shutdown status, and early-warning time, then audit forecast error.

**Mechanical Engineering:** Wait Versus Respond Early → **AI:** Intervention Simulation and Audit → `no action vs simulated speed reduction`

> 🎬 **See this illustrated and interactive:** [https://gearbox-vibration-prediction.streamlit.app/?stage=compare](https://gearbox-vibration-prediction.streamlit.app/?stage=compare)

## Part 4 · The technical explanation

In [ ]:
current=4.8;future=8.1;t=np.arange(31);without=current+(future-current)*(t/30)**1.35;with_ai=without.copy();with_ai[8:]-=np.linspace(0,future-6.2,23)
comparison=pd.DataFrame({"Metric":["Peak vibration","Time above threshold","Emergency shutdown","Early warning"],"Without AI":[f"{without.max():.1f} mm/s",f"{(without>7).sum()} s","Yes","0 s"],"With AI":[f"{with_ai.max():.1f} mm/s",f"{(with_ai>7).sum()} s","Avoided","22 s"]});display(comparison)
plt.plot(t,without,label="Without AI");plt.plot(t,with_ai,label="With simulated response");plt.axhline(7,color="red",ls="--");plt.xlabel("Seconds");plt.ylabel("Vibration mm/s");plt.grid(alpha=.2);plt.legend();plt.show()
actual_band=np.array([condition(v) for v in y_test]);pred_band=np.array([condition(v) for v in pred]);ConfusionMatrixDisplay(confusion_matrix(actual_band,pred_band,labels=["NORMAL","WARNING","DANGEROUS"]),display_labels=["Normal","Warning","Dangerous"]).plot(cmap="Blues");plt.show()
print("Limitations: synthetic waveform, simplified resonance/wear, no gearbox-specific limits, no sensor mounting study, no causal diagnosis, no validated control response.")

## Part 5 · What you just built

**In the notebook:** Generate the comparison table, confusion bands, MAE/RMSE, and limitations.

**Takeaway:** A simulated improvement is evidence for further study, not permission to control a real gearbox.

◀ [Previous: Project Condition Bands](https://gearbox-vibration-prediction.streamlit.app/?stage=decision) &nbsp;|&nbsp; [Project overview](https://gearbox-vibration-prediction.streamlit.app/?stage=start)

---
# Final engineering conclusion

The model fuses learned vibration-waveform patterns with current operating state to predict vibration 30 seconds ahead. A separate project rule turns that physical forecast into a simulated condition and response. Real machinery requires equipment-specific validation and approved controls.